# SIEE Shadow RGB: two-person YOLO11n v3

Choose Runtime → Change runtime type → T4 GPU. Run cells in order. This
experiment starts from the **saved v2 best.pt**, with a new optimizer run
(`resume=False`). Six operator-reviewed frames containing two people are
added to the original 230 training images. The 120-image validation set was
previously inspected and is only a development set. The six new images were
selected after inspecting v2 errors: their training score is diagnostic only.

No camera, UART or motor commands are run. Do not use the model to drive the
robot until a fresh independent two-person capture and separate range,
ownership, and stopping checks pass. If a source path differs in your Drive,
locate the same hash-verified file and adjust only FOLDER in cell 2.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import json
import math
import tarfile

FOLDER = Path('/content/drive/MyDrive/SIE/colab_runs/ar0234_shadow_yolo11n_v2')
PARTS = {
    's1': ('shadow_rgb_colab_v2_train_s1_20261031.tar.gz', '9a6a65a800c0847bdac646cd05c631de92aeacc976226160f9ec720ea03e4ebb', 238),
    's2': ('shadow_rgb_colab_v2_train_s2_20261031.tar.gz', '76550684e0826f210d93989a62f233443ae12af179066365e6bd53f38c191feb', 224),
    'val': ('shadow_rgb_colab_v2_val_20261031.tar.gz', '80b74757e35c0a76a7d13f21c43bc0f9ef3813c5af0f4156053c9a26ed706c6c', 240),
}
ADDENDUM = FOLDER / 'shadow_two_people_training_addendum_20261101.tar.gz'
ADDENDUM_SHA = '8e5038c5eb2c98150a5ab5c1110fb5ce69d111a0a7b939c76ec2175bf8445612'
V2_BEST = FOLDER / 'ar0234_shadow_person_upper_body_yolo11n_v2' / 'best.pt'
V2_BEST_SHA = '575c480b03b1722222211f474fae2dc8c7c6fe5e02c9078433d5d9e16667b395'
DATA_ROOT = Path('/content/sie_shadow_v3_two_people')
DATA_YAML = DATA_ROOT / 'dataset_v3.yaml'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

assert V2_BEST.is_file() and sha256(V2_BEST) == V2_BEST_SHA, 'V2_CHECKPOINT_MISMATCH'
assert ADDENDUM.is_file() and sha256(ADDENDUM) == ADDENDUM_SHA, 'ADDENDUM_SHA256_MISMATCH'
for name, (filename, expected, _) in PARTS.items():
    path = FOLDER / filename
    assert path.is_file() and sha256(path) == expected, f'DATA_ARCHIVE_SHA256_MISMATCH:{name}'
print('Verified v2 checkpoint, three v2 dataset parts, and reviewed addendum')


In [ ]:
# Keep the known training stack version; Colab supplies the CUDA-enabled PyTorch.
%pip install -q 'ultralytics==8.4.152' 'onnx>=1.17,<2'
import torch, ultralytics
print('torch', torch.__version__, 'ultralytics', ultralytics.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Reproduce the v2 pool, then add only six new training image/label pairs.
assert not DATA_ROOT.exists(), 'Rename the earlier /content/sie_shadow_v3_two_people run'
DATA_ROOT.mkdir()
extracted_names = set()
for name, (filename, _, expected_count) in PARTS.items():
    with tarfile.open(FOLDER / filename, 'r:gz') as tf:
        members = tf.getmembers()
        assert len(members) == expected_count, (name, len(members))
        assert all(m.isfile() and not m.name.startswith('/') and
                   '..' not in Path(m.name).parts and
                   m.name not in extracted_names for m in members)
        extracted_names.update(m.name for m in members)
        tf.extractall(DATA_ROOT, filter='data')
assert len(extracted_names) == 702
base = json.loads((DATA_ROOT / 'bundle_manifest.json').read_text())
assert base['sha256_overlap'] == 0
assert base['train']['images'] == 230 and base['val']['images'] == 120
for name, expected in base['per_image_sha256'].items():
    assert sha256(DATA_ROOT / name) == expected, name
base_hashes = set(base['per_image_sha256'].values())
assert len(base_hashes) == 350

with tarfile.open(ADDENDUM, 'r:gz') as tf:
    members = tf.getmembers()
    assert len(members) == 14 and all(m.isfile() and not m.name.startswith('/') and
                   '..' not in Path(m.name).parts for m in members)
    names = {m.name for m in members}
    assert len(names) == 14
    extra = json.load(tf.extractfile('manifest.json'))
    assert extra['status'] == 'OPERATOR_REVIEWED_TRAINING_ADDENDUM'
    assert extra['class_names'] == ['person_upper_body']
    assert extra['counts'] == {'images': 6, 'person_boxes': 12,
                               'base_train_images': 230, 'combined_train_images': 236}
    assert len(extra['rows']) == 6
    expected_names = {'manifest.json', 'README.md'}
    new_hashes = set()
    for row in extra['rows']:
        image = row['training_image']
        label = row['training_label']
        assert image.startswith('images/train/two_') and image.endswith('.jpg')
        assert label.startswith('labels/train/two_') and label.endswith('.txt')
        assert Path(image).stem == Path(label).stem and row['box_count'] == 2
        expected_names.update((image, label))
        assert not (DATA_ROOT / image).exists() and not (DATA_ROOT / label).exists()
        image_bytes = tf.extractfile(image).read()
        label_bytes = tf.extractfile(label).read()
        image_sha = hashlib.sha256(image_bytes).hexdigest()
        assert image_sha == row['source_image_sha256']
        assert hashlib.sha256(label_bytes).hexdigest() == row['training_label_sha256']
        assert image_sha not in base_hashes and image_sha not in new_hashes
        new_hashes.add(image_sha)
        lines = label_bytes.decode('ascii').strip().splitlines()
        assert len(lines) == 2
        for line in lines:
            values = line.split()
            assert values[0] == '0' and len(values) == 5
            cx, cy, w, h = map(float, values[1:])
            assert all(math.isfinite(x) for x in (cx, cy, w, h))
            assert 0 < w <= 1 and 0 < h <= 1
            # Nine-decimal YOLO rounding can turn a true edge of 0 into -5e-10.
            eps = 1e-8
            assert -eps <= cx-w/2 < cx+w/2 <= 1 + eps
            assert -eps <= cy-h/2 < cy+h/2 <= 1 + eps
    assert names == expected_names
    tf.extractall(DATA_ROOT, members=[m for m in members if m.name in
                  expected_names - {'manifest.json', 'README.md'}], filter='data')

for split, count in (('train', 236), ('val', 120)):
    images = sorted((DATA_ROOT / 'images' / split).glob('*.jpg'))
    labels = sorted((DATA_ROOT / 'labels' / split).glob('*.txt'))
    assert len(images) == len(labels) == count
    assert {p.stem for p in images} == {p.stem for p in labels}
DATA_YAML.write_text('path: ' + str(DATA_ROOT) + '\ntrain: images/train\nval: images/val\nnames:\n  0: person_upper_body\n')
print('Verified training 236 (including six two-person frames), development val 120')
print(DATA_YAML.read_text())


In [ ]:
from ultralytics import YOLO
import shutil

RUN_ROOT = Path('/content/runs')
RUN_NAME = 'ar0234_shadow_person_upper_body_yolo11n_v3_two_people'
DEST = FOLDER / RUN_NAME
assert not DEST.exists(), f'Choose a fresh run name: {DEST}'
model = YOLO(str(V2_BEST))
result = model.train(
    data=str(DATA_YAML), epochs=100, patience=20,
    imgsz=640, batch=16, seed=20260915,
    device=0, workers=2, optimizer='auto',
    project=str(RUN_ROOT), name=RUN_NAME, exist_ok=False,
    save_period=10, plots=True, resume=False,
)
RUN = Path(model.trainer.save_dir)
NEW_BEST = RUN / 'weights' / 'best.pt'
assert NEW_BEST.is_file()
DEST.mkdir()
shutil.copy2(NEW_BEST, DEST / 'best.pt')
shutil.copy2(RUN / 'args.yaml', DEST / 'args.yaml')
print('Saved new checkpoint:', DEST / 'best.pt', sha256(NEW_BEST))


In [ ]:
# Both checkpoints see the same inspected development validation.
old_model, new_model = YOLO(str(V2_BEST)), YOLO(str(NEW_BEST))
ap = {}
for label, net in (('v2', old_model), ('v3', new_model)):
    m = net.val(data=str(DATA_YAML), split='val', imgsz=640,
                device=0, plots=False, project=str(RUN_ROOT),
                name=RUN_NAME + '_' + label + '_dev_val')
    ap[label] = {'precision': float(m.box.mp), 'recall': float(m.box.mr),
                 'mAP50': float(m.box.map50), 'mAP50_95': float(m.box.map)}
print(json.dumps(ap, indent=2))


In [ ]:
# Frame-level presence and localization at the existing 0.40 operating threshold.
# This re-runs v2 via PyTorch, so it can differ from the earlier ONNX pipeline.
VAL_IMAGES = sorted((DATA_ROOT / 'images/val').glob('*.jpg'))
def overlap(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x1-x0) * max(0, y1-y0)
    aa = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    bb = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    return inter / (aa + bb - inter) if aa + bb > inter else 0.0

def frame_metrics(net):
    total = dict(positive=0, negative=0, tp_presence=0, tp_iou50=0, fp_images=0)
    for image in VAL_IMAGES:
        label = DATA_ROOT / 'labels/val' / (image.stem + '.txt')
        gt = label.read_text().strip()
        p = net.predict(source=str(image), imgsz=640, conf=0.40,
                        iou=0.45, device=0, verbose=False)[0]
        boxes = p.boxes.xyxy.cpu().numpy().tolist()
        if not gt:
            total['negative'] += 1
            total['fp_images'] += bool(boxes)
            continue
        total['positive'] += 1
        class_id, cx, cy, w, h = map(float, gt.split())
        assert class_id == 0
        iw, ih = p.orig_shape[1], p.orig_shape[0]
        actual = ((cx-w/2)*iw, (cy-h/2)*ih,
                  (cx+w/2)*iw, (cy+h/2)*ih)
        total['tp_presence'] += bool(boxes)
        total['tp_iou50'] += any(overlap(box, actual) >= 0.50 for box in boxes)
    return total

frames = {'v1': frame_metrics(old_model), 'v2': frame_metrics(new_model)}
print(json.dumps(frames, indent=2))


In [ ]:
# Training-frame diagnostic only: two distinct matches per image.
from itertools import permutations
def two_person_matches(net):
    passed = 0
    for row in extra['rows']:
        image = DATA_ROOT / row['training_image']
        labels = (DATA_ROOT / row['training_label']).read_text().strip().splitlines()
        predicted = net.predict(source=str(image), imgsz=640, conf=0.40,
                                iou=0.45, device=0, verbose=False)[0]
        iw, ih = predicted.orig_shape[1], predicted.orig_shape[0]
        gt = []
        for line in labels:
            class_id, cx, cy, w, h = map(float, line.split())
            assert class_id == 0
            gt.append(((cx-w/2)*iw, (cy-h/2)*ih,
                       (cx+w/2)*iw, (cy+h/2)*ih))
        boxes = predicted.boxes.xyxy.cpu().numpy().tolist()
        passed += any(all(overlap(boxes[i], gt[j]) >= 0.5 for j, i in enumerate(pair))
                      for pair in permutations(range(len(boxes)), 2))
    return passed

training_six = {'v2': two_person_matches(old_model),
                'v3': two_person_matches(new_model)}
print('Six selected training images, both boxes IoU >= 0.50:', training_six)
print('These are training examples, not an independent evaluation.')

In [ ]:
# Export static ONNX with the v2 input contract; verify on Pi separately.
import onnx
exported = Path(YOLO(str(NEW_BEST)).export(
    format='onnx', imgsz=640, dynamic=False, simplify=False,
    opset=17, nms=False, device=0,
))
onnx.checker.check_model(onnx.load(str(exported)))
shutil.copy2(exported, DEST / 'best.onnx')
summary = {
    'scope': 'SHADOW_RGB_DEVELOPMENT_ONLY',
    'dataset_parts_sha256': {name: digest for name, (_, digest, _) in PARTS.items()},
    'addendum_sha256': ADDENDUM_SHA,
    'addendum_source_archive_sha256': extra['reviewed_archive_sha256'],
    'v2_checkpoint_sha256': V2_BEST_SHA,
    'v3_checkpoint_sha256': sha256(NEW_BEST),
    'v3_onnx_sha256': sha256(exported),
    'ultralytics_version': ultralytics.__version__,
    'torch_version': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'ap': ap, 'frames_conf_0p40_iou_0p50': frames,
    'training_six_both_boxes_iou_0p50': training_six,
    'limits': 'Six selected failures are now training data. The 120-image validation was previously inspected. Collect a fresh blind two-person capture before any motion use.',
}
(DEST / 'comparison.json').write_text(json.dumps(summary, indent=2) + chr(10))
print('Saved:', DEST)
print(json.dumps(summary, indent=2))
